In [2]:
!git clone https://github.com/campusx-official/rag-eval-deepeval.git

Cloning into 'rag-eval-deepeval'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 127 (delta 32), reused 31 (delta 31), pack-reused 90 (from 1)
Receiving objects: 100% (127/127), 667.87 KiB | 4.70 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [5]:
%cd rag-eval-deepeval

/content/rag-eval-deepeval


In [6]:
!pip install -q langchain langchain-openai langchain-chroma deepeval python-dotenv langsmith

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 892.2/892.2 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.

In [10]:
!pip install -q langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.8 MB/s eta 0:00:00


In [ ]:
import os

# Set your Groq API key directly
os.environ["GROQ_API_KEY"] = "API_KEY_HERE"

# Create directories for source code and data
os.makedirs("src", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("chroma_store", exist_ok=True)

In [ ]:
%%writefile evals/eval_generator.py
from dotenv import load_dotenv
import os
import time

from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.models import LiteLLMModel

from src.generator import generate
from evals.harness import load_goldens, summarize_by_metric, print_summary

load_dotenv()

GOLDEN_PATH = "goldens/faithfulness_dataset.json"
THRESHOLD = 0.7

os.environ["GROQ_API_KEY"] = "API_KEY_HERE"

# Configure LiteLLM for Groq with enough tokens to return valid JSON schemas
judge_llm = LiteLLMModel(
    model="groq/qwen/qwen3.8-27b",
    max_tokens=1024
)

# A simple dummy class to match the attribute lookup of summarize_by_metric
class DummyTestResult:
    def __init__(self, metrics):
        self.metrics = metrics

def run():
    # Slice dataset to exactly 3 test cases to strictly stay under 1,000 OTPM limit
    goldens = load_goldens(GOLDEN_PATH)[:3]

    # Initialize metrics
    faithfulness_metric = FaithfulnessMetric(
        threshold=THRESHOLD,
        model=judge_llm,
        include_reason=True,
        async_mode=False
    )
    relevancy_metric = AnswerRelevancyMetric(
        threshold=THRESHOLD,
        model=judge_llm,
        include_reason=True,
        async_mode=False
    )

    results_list = []

    print(f"Evaluating {len(goldens)} test cases sequentially to respect Groq rate limits...")
    for idx, g in enumerate(goldens):
        print(f"Processing test case #{idx + 1}/{len(goldens)}...")

        # Generate answer sequentially
        context = g["ideal_context"]
        answer = generate(g["query"], context)

        test_case = LLMTestCase(
            input=g["query"],
            actual_output=answer,
            retrieval_context=context,
        )

        # Execute metrics sequentially with 15s pacing to allow Groq's OTPM window to reset
        try:
            time.sleep(15)
            faithfulness_metric.measure(test_case)

            time.sleep(15)
            relevancy_metric.measure(test_case)

            # We map the exact attribute names as expected by summarize_by_metric
            from types import SimpleNamespace
            m1 = SimpleNamespace(
                name="Faithfulness",
                score=faithfulness_metric.score,
                success=faithfulness_metric.is_successful()
            )
            m2 = SimpleNamespace(
                name="Answer Relevancy",
                score=relevancy_metric.score,
                success=relevancy_metric.is_successful()
            )

            results_list.append(DummyTestResult([m1, m2]))
        except Exception as e:
            print(f"Error evaluating test case #{idx + 1}: {e}")
            time.sleep(15)  # Generous backoff on failure

    # Format results to match the summary structure
    return summarize_by_metric(results_list)

if __name__ == "__main__":
    print_summary("generator", run())

Overwriting evals/eval_generator.py


In [36]:
%%writefile src/retriever.py
import os
import re
import glob

from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

load_dotenv()

DATA_DIR = "data"
DB_DIR = "chroma_store"

def load_transcripts():
    docs = []
    for path in glob.glob(f"{DATA_DIR}/*.vtt"):
        lines = []
        for line in open(path):
            line = line.strip()
            if not line or line == "WEBVTT" or "-->" in line:
                continue
            lines.append(line)
        text = " ".join(lines)

        match = re.search(r"Session[ _]*(\d+)", path, re.IGNORECASE)
        session = match.group(1) if match else "1"

        docs.append(Document(page_content=text, metadata={"session": session}))

    return docs

def load_store():
    # Use free, local Hugging Face embeddings instead of OpenAI
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    if os.path.exists(DB_DIR) and os.listdir(DB_DIR):
        return Chroma(persist_directory=DB_DIR, embedding_function=embeddings)

    docs = load_transcripts()
    if not docs:
        raise ValueError("No transcript documents found in data/ directory!")

    chunks = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100,
    ).split_documents(docs)

    return Chroma.from_documents(chunks, embeddings, persist_directory=DB_DIR)

def build_retriever():
    return load_store().as_retriever(search_kwargs={"k": 2})

if __name__ == "__main__":
    retriever = build_retriever()
    results = retriever.invoke("what is regression testing?")
    for r in results:
        print(f"[Session {r.metadata['session']}] {r.page_content[:150]}...\n")

Overwriting src/retriever.py


In [46]:
%%writefile src/generator.py
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    max_tokens=50,  # Strict limit to prevent Groq 429 OTPM errors
    api_key=os.environ.get("GROQ_API_KEY")
)

prompt = ChatPromptTemplate.from_template(
    """
You are a helpful teaching assistant for a course on LLM evaluations. Answer the student's question using ONLY the information in the context provided below.

Rules:
- Use only information present in the context. Do not add outside knowledge.
- Answer thoroughly: identify every distinct part of the question and cover each one, and include all the relevant points the context provides for answering it.
- If the context does not contain enough information to answer, say exactly:
"I don't have enough information in the course material to answer that."

<COURSE_CONTEXT>
{context}
</COURSE_CONTEXT>

<STUDENT_QUESTION>
{question}
</STUDENT_QUESTION>

Answer:
"""
)

chain = prompt | llm | StrOutputParser()

def generate(query: str, context: list[str]) -> str:
    """Generate a grounded answer from the query and context chunks."""
    context_text = "\n\n".join(context)
    return chain.invoke({"question": query, "context": context_text})

Overwriting src/generator.py


In [38]:
!pip install -q litellm

In [58]:
!python -m evals.eval_generator

Evaluating 3 test cases sequentially to respect Groq rate limits...
Processing test case #1/3...
✨ You're running DeepEval's latest Faithfulness Metric! (using groq/qwen/qwen3…
✨ You're running DeepEval's latest Answer Relevancy Metric! (using groq/qwen/q…
Processing test case #2/3...
✨ You're running DeepEval's latest Faithfulness Metric! (using groq/qwen/qwen3…
✨ You're running DeepEval's latest Answer Relevancy Metric! (using groq/qwen/q…
Processing test case #3/3...
✨ You're running DeepEval's latest Faithfulness Metric! (using groq/qwen/qwen3…
✨ You're running DeepEval's latest Answer Relevancy Metric! (using groq/qwen/q…

generator  (per-metric summary)
  Faithfulness               pass_rate=  100%  avg=1.00  n=3
  Answer Relevancy           pass_rate=   33%  avg=0.56  n=3


In [ ]:
%%writefile evals/eval_pipeline.py
from dotenv import load_dotenv
import os
import time

from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.models import LiteLLMModel

from src.retriever import build_retriever
from src.generator import generate
from evals.harness import load_goldens, summarize_by_metric, print_summary

load_dotenv()

GOLDEN_PATH = "goldens/faithfulness_dataset.json"
THRESHOLD = 0.7

os.environ["GROQ_API_KEY"] = "API_HERE"

# Configure LiteLLM for Groq with max_tokens=512 to stay strictly below the 1000 OTPM limit
judge_llm = LiteLLMModel(
    model="groq/qwen/qwen3.8-27b",
    max_tokens=512
)

# A simple dummy class to match the attribute lookup of summarize_by_metric
class DummyTestResult:
    def __init__(self, metrics):
        self.metrics = metrics

def run():
    # Slice dataset to exactly 3 test cases to stay under Groq's rate limits
    goldens = load_goldens(GOLDEN_PATH)[:3]

    # Initialize our retriever
    print("Initializing Chroma vector store retriever...")
    retriever = build_retriever()

    # Initialize metrics
    faithfulness_metric = FaithfulnessMetric(
        threshold=THRESHOLD,
        model=judge_llm,
        include_reason=True,
        async_mode=False
    )
    relevancy_metric = AnswerRelevancyMetric(
        threshold=THRESHOLD,
        model=judge_llm,
        include_reason=True,
        async_mode=False
    )

    results_list = []

    print(f"Evaluating {len(goldens)} pipeline test cases sequentially...")
    for idx, g in enumerate(goldens):
        print(f"\n--- Processing test case #{idx + 1}/{len(goldens)} ---")
        print(f"Query: {g['query']}")

        # 1. Retrieve context chunks dynamically from our retriever
        retrieved_docs = retriever.invoke(g["query"])
        context_chunks = [doc.page_content for doc in retrieved_docs]

        # 2. Generate answer sequentially
        answer = generate(g["query"], context_chunks)
        print(f"Generated Answer: {answer[:100]}...")

        test_case = LLMTestCase(
            input=g["query"],
            actual_output=answer,
            retrieval_context=context_chunks,
        )

        # 3. Execute metrics sequentially with pacing to avoid Rate Limit Errors
        try:
            time.sleep(15)
            faithfulness_metric.measure(test_case)

            time.sleep(15)
            relevancy_metric.measure(test_case)

            # Map attributes to format expected by summarize_by_metric
            from types import SimpleNamespace
            m1 = SimpleNamespace(
                name="Faithfulness",
                score=faithfulness_metric.score,
                success=faithfulness_metric.is_successful()
            )
            m2 = SimpleNamespace(
                name="Answer Relevancy",
                score=relevancy_metric.score,
                success=relevancy_metric.is_successful()
            )

            results_list.append(DummyTestResult([m1, m2]))
        except Exception as e:
            print(f"Error evaluating test case #{idx + 1}: {e}")
            time.sleep(15)

    return summarize_by_metric(results_list)

if __name__ == "__main__":
    print_summary("full_pipeline", run())

Overwriting evals/eval_pipeline.py


In [71]:
!python -m evals.eval_pipeline

Initializing Chroma vector store retriever...
Loading weights: 100% 103/103 [00:00<00:00, 4516.84it/s]
Evaluating 3 pipeline test cases sequentially...

--- Processing test case #1/3 ---
Query: what is vibe testing and why isn't it enough for deploying an llm app?
Generated Answer: Based on the provided course context, here is the answer to your question:

**What is vibe testing?*...
✨ You're running DeepEval's latest Faithfulness Metric! (using groq/qwen/qwen3…
✨ You're running DeepEval's latest Answer Relevancy Metric! (using groq/qwen/q…

--- Processing test case #2/3 ---
Query: what is the mmlu benchmark?
Generated Answer: Based on the provided course context, the MMLU benchmark is described as follows:

*   **Scope:** It...
✨ You're running DeepEval's latest Faithfulness Metric! (using groq/qwen/qwen3…
✨ You're running DeepEval's latest Answer Relevancy Metric! (using groq/qwen/q…

--- Processing test case #3/3 ---
Query: how did mmlu pro improve on the original mmlu?
Generated An

In [68]:
!pip install --upgrade -q langchain-huggingface sentence-transformers langchain-text-splitters

### Quick Analysis of Your RAG Pipeline:
Faithfulness (Pass Rate: 67%, Avg: 0.83): This measures how grounded your generator's answers are in the retrieved contexts. An average score of 0.83 is quite strong, meaning that for most questions, the generator is sticking closely to the retrieved course content without hallucinating.
### Answer Relevancy (Pass Rate: 33%, Avg: 0.33): This measures how directly the generated answer addresses the user's query. The lower score here suggests that while the answer is faithful to the context, the dynamic contexts fetched by your retriever might not have fully matched the queries, or the generated answer was slightly off-topic relative to the specific question asked. This gives you a clear optimization target (e.g., tuning chunk sizes, adjusting top-k retrieval parameters, or refining system prompts).